In [1]:
import pandas as pd
from pathlib import Path
from utility_functions import FEATURE_COLS, TRAIN_SEASONS, VAL_SEASONS, load_data, score

CURRENT_DIR = Path.cwd()
train, holdout = load_data(CURRENT_DIR)

print(f'train  : {len(train)//2:,} matches  seasons {train.season.min()}-{train.season.max()}')
print(f'holdout: {len(holdout)//2:,} matches  dates {holdout.date.min().date()} to {holdout.date.max().date()}')
print()
train.head(3)


train  : 2,090 matches  seasons 2012-2022
holdout: 760 matches  dates 2023-08-11 to 2027-05-14



,match_id,date,season,team,opponent,home,corners,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10
0,M2012001,2012-08-10,2012,team_01,team_02,1,6,0.716,0.613,0.556,0.975,0.551,0.351,0.119,0.302,0.483,0.727
1,M2012001,2012-08-10,2012,team_02,team_01,0,10,0.451,0.657,0.936,0.464,0.500,0.858,0.778,0.257,0.872,0.698
2,M2012002,2012-08-12,2012,team_01,team_03,1,10,0.797,0.479,0.516,0.952,0.898,0.385,0.405,0.248,0.509,0.577


### Target: corners distribution

Corners are count data (non-negative integers).
We want to know: roughly Poisson, or wider?  Do home teams get more?

In [2]:
c = train['corners']
print(f'mean: {c.mean():.2f}')
print(f'std: {c.std():.2f}')
print(f'var/mean : {c.var()/c.mean():.2f}: Poisson = 1.0, NegBin > 1.0')

home_c = train.loc[train['home'] == 1, 'corners']
away_c = train.loc[train['home'] == 0, 'corners']
print(f'home avg: {home_c.mean():.3f}   away avg: {away_c.mean():.3f}')

mean: 6.03
std: 3.33
var/mean : 1.84: Poisson = 1.0, NegBin > 1.0
home avg: 6.643   away avg: 5.410


### Features: range and correlation with corners

In [3]:
# Sort using corelation with target variable
corr = train[FEATURE_COLS].corrwith(train['corners']).sort_values(key=abs, ascending=False)
print('Correlation with corners:')
print(corr.round(4))

Correlation with corners:
feature_8     0.3620
feature_7     0.2613
feature_4     0.1462
feature_3     0.1051
feature_9     0.1013
feature_5    -0.0312
feature_6    -0.0135
feature_2    -0.0111
feature_10   -0.0056
feature_1     0.0024
dtype: float64


### Train / validation split

- **Train** 2012–2019 — what the model learns from
- **Validation** 2020–2022 — compare against baseline here
- **Holdout** 2023–2026 — no labels, this is what we predict

In [4]:
train_only = train[train['season'].isin(TRAIN_SEASONS)]
val        = train[train['season'].isin(VAL_SEASONS)]

print(f'Train  : {len(train_only)//2:,} matches  ({TRAIN_SEASONS[0]}-{TRAIN_SEASONS[-1]})')
print(f'Val    : {len(val)//2:,} matches  ({VAL_SEASONS})')
print(f'Holdout: {len(holdout)//2:,} matches  (2023-2026)')


Train  : 1,520 matches  (2012-2019)
Val    : 570 matches  ([2020, 2021, 2022])
Holdout: 760 matches  (2023-2026)


### Baseline evaluation

In [5]:
baseline = pd.read_csv(CURRENT_DIR / 'inputs' / 'baseline_validation_predictions.csv')
val_b    = val.merge(baseline, on=['match_id', 'team'])

y    = val_b['corners'].to_numpy(dtype=float)
yhat = val_b['baseline_predicted_corners'].to_numpy(dtype=float)

score('Baseline', y, yhat, width=20)

Baseline             MAE=2.2717  RMSE=2.8574  PoissonDev=1.3406


### Linear regression & Poisson GLM

In [6]:
from sklearn.linear_model import LinearRegression, PoissonRegressor

MODEL_COLS = FEATURE_COLS + ['home']
X_train = train_only[MODEL_COLS].to_numpy()
X_val   = val[MODEL_COLS].to_numpy()
y_val   = val['corners'].to_numpy(dtype=float)

lr = LinearRegression().fit(X_train, train_only['corners'])
pg = PoissonRegressor(alpha=0, max_iter=300).fit(X_train, train_only['corners'])

score('Baseline',          y_val, yhat, width=20)
score('Linear Regression', y_val, lr.predict(X_val), width=20)
score('Poisson GLM',       y_val, pg.predict(X_val), width=20)

Baseline             MAE=2.2717  RMSE=2.8574  PoissonDev=1.3406
Linear Regression    MAE=2.2535  RMSE=2.8237  PoissonDev=1.3421
Poisson GLM          MAE=2.2717  RMSE=2.8574  PoissonDev=1.3406


#### Conclusions
- **Baseline = Poisson GLM** (home + 10 features): metrics identical.
- **`home` flag is essential**: without it all three models underperform baseline; adding it closes the gap entirely